In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "/workspaces/dev/test/performance_test/libri",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path

In [ ]:
from sj_utils.collection import SafetyDict
from sj_ai_utils.datasets.libri_speech_asr_corpus.service import search_dirs

In [ ]:
from util import rt_whisper

In [ ]:
RANDOM_SEED = 42
CHUNK_SIZE = 48000
USE_SAVE_LOADER = False
USE_PROMPT = True
TEST_ALL = True
MAX_COUNT = 3

In [ ]:
SOURCE = "/workspaces/dev/.datasets/LibriSpeechASRcorpus/dev-clean"
STORAGE = "/workspaces/dev/.storage/libri/"
HYPERPARAMETERS_PATH = "/workspaces/dev/test/optimize/esic/hyperparameters/20250731/step1_16b-96k/trial_wer3o0_6690_20250801_074742.yaml"

In [ ]:
source = Path(SOURCE)
storage = Path(STORAGE)
hyperparameter_path = Path(HYPERPARAMETERS_PATH)

if not source.exists():
    raise FileNotFoundError(f"Source path does not exist: {source}")
if not hyperparameter_path.exists():
    raise FileNotFoundError(f"Hyperparameter path does not exist: {hyperparameter_path}")

In [ ]:
data_paths = search_dirs(source)

In [ ]:
hyperparameter = SafetyDict({
    "whisper": {
        "model_options": {
            "model_size_or_path": "large-v3",
            "device": "cuda",
            "compute_type": "float16",
        },
        "transcribe_options": {
            "beam_size":5,
            "vad_filter": False,
            "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        }
    },
    "silero_vad": {
        "model_options": {},
        "run_options": {}
    },
    "asr": {
        "max_overlap_duration": 16000,
    },
    "position_weighted_filter": {
        "boundary": 0,
    },
    "duration_filter": {
        "z_thresh": {
            "default": 2.0,
            "ko": 2.0,
            "en": 5.0,
        },
        "min_dur": {
            "default": 160,
            "ko": 160,
            "en": 160,
        }
    },
    "probability_filter":{
        "z_thresh":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.4,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.15
        },
    },
    "selector":{
        "iou_threshold": {
            "default": 0.5,
            "ko": 0.4,
            "en": 0.75,
        },
        "cos_threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.64
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 15800
        },
    },
})

In [ ]:
rt_whisper(
    source,
    storage,
    data_paths,
    seed = RANDOM_SEED,
    use_save_loader=USE_SAVE_LOADER,
    use_prompt=USE_PROMPT,
    # hyperparameter=hyperparameter,
    hyperparameter=hyperparameter_path,
    chunk_size=CHUNK_SIZE,
    test_all = TEST_ALL,
    max_count=MAX_COUNT
)